# 07 - LLM, Prompt, Chain Và History

Notebook này dùng để **lắp prompt, chain và lịch sử hội thoại**. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.

### BƯỚC 1: Setup Tối Thiểu

- **Tác dụng chính:** Notebook này dùng để lắp prompt, chain và lịch sử hội thoại. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `find_chatbot_fashion_root`, `CHATBOT_FASHION_DIR` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [1]:
import sys

from pathlib import Path

import ollama

from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_community.chat_message_histories import RedisChatMessageHistory

from langchain_core.documents import Document

from langchain_core.messages import BaseMessage, SystemMessage

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate

from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_ollama import ChatOllama

def find_chatbot_fashion_root(start: Path | None = None) -> Path:
    """Tìm thư mục dự án có chứa `app/config.py`.

    Args:
        start (Path | None): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        Path: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        for candidate in (parent, parent / "Chatbot_Fashion"):
            if (candidate / "app" / "config.py").exists():
                return candidate
    raise RuntimeError("Không tìm thấy thư mục gốc Chatbot_Fashion chứa app/config.py")


In [2]:
CHATBOT_FASHION_DIR = find_chatbot_fashion_root()

if str(CHATBOT_FASHION_DIR) not in sys.path:
    sys.path.insert(0, str(CHATBOT_FASHION_DIR))

from app.config import (
    HISTORY_MAX_MESSAGES,
    HISTORY_RECENT_KEEP,
    LLM_MODEL,
    LLM_NUM_CTX,
    LLM_NUM_PREDICT,
    LLM_TEMPERATURE,
    LLM_TIMEOUT,
    OLLAMA_BASE_URL,
    REDIS_URL,
    SUMMARIZE_MAX_TOKENS,
)

print("[OK] Setup notebook 07 hoàn tất")

print(f"Project root: {CHATBOT_FASHION_DIR}")


[OK] Setup notebook 07 hoàn tất
Project root: D:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion


### BƯỚC 2: Khởi Tạo LLM Client

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `llm` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [3]:
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=LLM_TEMPERATURE,
    timeout=LLM_TIMEOUT,
    num_predict=LLM_NUM_PREDICT,
    num_ctx=LLM_NUM_CTX,
)

print("[OK] Đã tạo ChatOllama client; chưa gọi model")
print(f"Model       : {LLM_MODEL}")
print(f"Ollama URL  : {OLLAMA_BASE_URL}")
print(f"Temperature : {LLM_TEMPERATURE}")
print(f"Context     : {LLM_NUM_CTX} tokens")
print(f"Output tối đa: {LLM_NUM_PREDICT} tokens")


[OK] Đã tạo ChatOllama client; chưa gọi model
Model       : qwen3:4b-instruct
Ollama URL  : http://localhost:11434
Temperature : 0.4
Context     : 4096 tokens
Output tối đa: 1024 tokens


### BƯỚC 3: Prompt Cho Luồng Tìm Sản Phẩm

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `SEARCH_SYSTEM_PROMPT`, `format_documents_for_llm`, `QA_PROMPT`, `contextualize_q_prompt`, `doc_prompt` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [4]:
SEARCH_SYSTEM_PROMPT = (
    "Bạn là chuyên viên tư vấn thời trang cao cấp, thân thiện và nói tiếng Việt tự nhiên.\n\n"
    "QUY TẮC TỐI CAO:\n"
    "1. Chỉ dùng thông tin có trong phần \"DỮ LIỆU SẢN PHẨM\". Không bịa tên, mã, giá, ảnh hoặc đặc điểm.\n"
    "2. Giới thiệu tối đa 5 sản phẩm. Nếu context có nhiều hơn, chọn 5 sản phẩm phù hợp nhất.\n"
    "3. Toàn bộ câu trả lời không vượt quá 400 từ.\n"
    "4. Trước khi trả lời, tự kiểm tra sản phẩm có đúng nhu cầu không. Không trình bày quá trình suy luận.\n"
    "5. Kết thúc bằng đúng 1 câu hỏi gợi mở để tiếp tục tư vấn.\n\n"
    "SCHEMA BẮT BUỘC:\n"
    "Mình gợi ý cho bạn tối đa 5 lựa chọn sau:\n\n"
    "1. **Tên sản phẩm**\n"
    "- Mã SP: [MÃ_SP]\n"
    "- Giá: [GIÁ] VND\n"
    "- Đặc điểm: [màu/chất liệu/kiểu dáng/dịp mặc nổi bật]\n"
    "- Lý do phù hợp: [1 câu ngắn, dựa trên yêu cầu của khách]\n"
    "- Ảnh: ![Sản phẩm]([IMAGE_URL])\n\n"
    "Nếu không có ảnh, ghi: \"Ảnh: Chưa có ảnh\".\n"
    "Nếu không có sản phẩm phù hợp trong context, xin lỗi ngắn gọn và hỏi khách có muốn đổi phong cách không.\n\n"
    "DỮ LIỆU SẢN PHẨM:\n"
    "{context}"
)

def format_documents_for_llm(docs: list[Document]) -> str:
    """Chuyển danh sách sản phẩm thành một khối context có cấu trúc.

    Args:
        docs (list[Document]): Document hoặc danh sách Document cần xử lý.

    Returns:
        str: Kết quả đã xử lý của hàm.
    """
    # Import trì hoãn để các cell học prompt phía trên không phải tải sớm
    # toàn bộ module vector store và các thư viện model liên quan.
    from app.core.vector_store import normalize_product_metadata

    formatted_documents: list[str] = []

    for doc in docs:
        normalized_doc = normalize_product_metadata(doc)
        formatted_documents.append(
            doc_prompt.format(
                product_id=normalized_doc.metadata.get("product_id", "N/A"),
                image_url=normalized_doc.metadata.get("image_url", ""),
                page_content=normalized_doc.page_content,
            )
        )

    return "\n".join(formatted_documents)


In [5]:
QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SEARCH_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Nhiệm vụ của bạn là VIẾT LẠI CÂU HỎI.\n"
        "Dựa vào lịch sử trò chuyện, hãy làm rõ nghĩa của câu hỏi mới nhất "
        "để nó có thể đứng độc lập mà ai đọc cũng hiểu được.\n\n"
        "QUY TẮC SỐNG CÒN:\n"
        "- TUYỆT ĐỐI KHÔNG TRẢ LỜI CÂU HỎI CỦA KHÁCH.\n"
        "- CHỈ IN RA DUY NHẤT CÂU HỎI ĐÃ ĐƯỢC VIẾT LẠI. Không giải thích, không dạ thưa.\n"
        "- Nếu câu hỏi đã rõ ràng, hãy in lại nguyên câu đó.\n\n"
        "VÍ DỤ:\n"
        "Khách: \"Có màu khác không?\" -> CHỈ IN RA: \"Áo thun đỏ ở trên có màu khác không?\"",
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

doc_prompt = PromptTemplate.from_template(
    "\n[MÃ_SP: {product_id}]"
    "\nIMAGE_URL: {image_url}"
    "\nTHÔNG TIN CHI TIẾT: {page_content}\n"
)

print("[OK] Đã khai báo prompt tìm kiếm và bộ định dạng Document")


[OK] Đã khai báo prompt tìm kiếm và bộ định dạng Document


### BƯỚC 4: Prompt Cho Luồng Phối Đồ

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `OUTFIT_SYSTEM_PROMPT`, `outfit_prompt` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [6]:
OUTFIT_SYSTEM_PROMPT = (
    "Bạn là chuyên gia tạo dáng (Personal Stylist) chuyên nghiệp và tâm lý.\n\n"
    "NHIỆM VỤ: Dựa vào \"CÔNG THỨC PHỐI ĐỒ\" và \"SẢN PHẨM GỢI Ý\" bên dưới, "
    "tạo một outfit hoàn chỉnh cho khách.\n\n"
    "QUY TẮC:\n"
    "1. Chỉ giới thiệu sản phẩm có trong \"SẢN PHẨM GỢI Ý\". Không thêm món ngoài context.\n"
    "2. Tối đa 3 sản phẩm, không vượt quá 400 từ.\n"
    "3. Tự kiểm tra sự hài hòa màu sắc, bối cảnh sử dụng và vóc dáng/tone da nếu có.\n"
    "4. Không trình bày quá trình suy luận.\n"
    "5. Kết thúc bằng đúng 1 câu hỏi gợi mở để tiếp tục tư vấn.\n\n"
    "SCHEMA BẮT BUỘC:\n"
    "Mình phối cho bạn một set như sau:\n\n"
    "1. **Tên sản phẩm**\n"
    "- Mã SP: [MÃ_SP]\n"
    "- Giá: [GIÁ] VND\n"
    "- Đặc điểm: [màu/chất liệu/kiểu dáng nổi bật]\n"
    "- Lý do phù hợp: [1 câu ngắn, gắn với công thức phối đồ]\n"
    "- Ảnh: ![Sản phẩm]([IMAGE_URL])\n\n"
    "Nếu không có ảnh, ghi: \"Ảnh: Chưa có ảnh\".\n\n"
    "{outfit_context}"
)


In [7]:
outfit_prompt = ChatPromptTemplate.from_messages([
    ("system", OUTFIT_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

print("[OK] Đã khai báo prompt phối đồ")


[OK] Đã khai báo prompt phối đồ


### BƯỚC 5: Redis History Và Tóm Tắt Hội Thoại

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `SUMMARIZE_PROMPT`, `ollama_client` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [8]:
SUMMARIZE_PROMPT = (
    "Tóm tắt cuộc hội thoại mua sắm thời trang sau thành 3-5 câu ngắn.\n"
    "Giữ lại: sản phẩm đã hỏi, phong cách khách thích, thông tin vóc dáng/tone da (nếu có).\n"
    "Bỏ qua: lời chào và câu xã giao.\n"
    "Chỉ trả về đoạn tóm tắt, không thêm nội dung khác.\n\n"
    "HỘI THOẠI:\n"
    "{history_text}"
)


In [9]:
ollama_client = ollama.Client(host=OLLAMA_BASE_URL)

print(f"Ngưỡng tóm tắt : trên {HISTORY_MAX_MESSAGES} messages")

print(f"Giữ nguyên      : {HISTORY_RECENT_KEEP} messages gần nhất")

print(f"Token tóm tắt  : tối đa {SUMMARIZE_MAX_TOKENS}")


Ngưỡng tóm tắt : trên 8 messages
Giữ nguyên      : 4 messages gần nhất
Token tóm tắt  : tối đa 150


### BƯỚC 5B: Biến Message Thành Bản Tóm Tắt

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `_message_speaker`, `summarize_history` để các bước sau tiếp tục sử dụng.


In [10]:
def _message_speaker(message: BaseMessage) -> str:
    """Đổi loại message của LangChain thành nhãn tiếng Việt dễ đọc.

    Args:
        message (BaseMessage): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        str: Kết quả đã xử lý của hàm.
    """
    speaker_by_type = {
        "human": "Khách",
        "ai": "Chatbot",
        "system": "Tóm tắt trước",
    }
    return speaker_by_type.get(message.type, message.type)


def summarize_history(messages: list[BaseMessage]) -> str:
    """Dùng LLM nén các message cũ thành một đoạn tóm tắt ngắn.

    Args:
        messages (list[BaseMessage]): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        str: Kết quả đã xử lý của hàm.
    """
    history_lines = [
        f"{_message_speaker(message)}: {str(message.content)[:300]}"
        for message in messages
    ]
    history_text = "\n".join(history_lines)

    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{
            "role": "user",
            "content": SUMMARIZE_PROMPT.format(history_text=history_text),
        }],
        options={
            "temperature": 0,
            "num_predict": SUMMARIZE_MAX_TOKENS,
        },
    )
    return response["message"]["content"].strip()


### BƯỚC 5C: Đọc Và Nén History Theo session_id

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `get_message_history` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [11]:
def get_message_history(session_id: str) -> RedisChatMessageHistory:
    """Lấy lịch sử Redis và tự nén khi số message vượt ngưỡng.

    Args:
        session_id (str): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        RedisChatMessageHistory: Kết quả đã xử lý của hàm.
    """
    history = RedisChatMessageHistory(session_id=session_id, url=REDIS_URL)
    messages = history.messages

    if len(messages) <= HISTORY_MAX_MESSAGES:
        return history

    old_messages = messages[:-HISTORY_RECENT_KEEP]
    recent_messages = messages[-HISTORY_RECENT_KEEP:]
    summary_text = summarize_history(old_messages)

    history.clear()
    history.add_message(SystemMessage(
        content=f"[TÓM TẮT HỘI THOẠI TRƯỚC]: {summary_text}",
    ))
    history.add_messages(recent_messages)
    return history


In [12]:
print("[OK] Đã khai báo lớp history; chưa kết nối Redis")


[OK] Đã khai báo lớp history; chưa kết nối Redis


### BƯỚC 6: Lắp Ráp Ba Chuỗi Xử Lý

- **Tác dụng chính:** Thực hiện bước lắp prompt, chain và lịch sử hội thoại.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `retriever`, `history_aware_retriever`, `document_chain`, `rag_chain`, `full_chat_chain` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [13]:
print("[THÔNG BÁO] Đang kết nối retriever và lắp ráp các chain...")

# Chain 1: RAG đầy đủ cho tìm kiếm bằng văn bản.
# History-aware retriever dùng LLM để viết lại câu hỏi trước khi search.

from app.core.vector_store import get_product_retriever

retriever = get_product_retriever()
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt,
)

# Stuff documents chain định dạng từng Document bằng `doc_prompt`,
# ghép chúng vào `{context}`, rồi mới gửi prompt hoàn chỉnh cho LLM.
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=QA_PROMPT,
    document_prompt=doc_prompt,
)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

full_chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# Chain 2: image retrieval đã hoàn tất ở bên ngoài. Caller phải dùng
# `format_documents_for_llm()` để tạo chuỗi `{context}` trước khi gọi chain.

product_answer_llm_chain = QA_PROMPT | llm

product_answer_chain_with_history = RunnableWithMessageHistory(
    product_answer_llm_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# Chain 3: Layer A và Layer B đã được ghép thành `{outfit_context}` bên ngoài.

outfit_llm_chain = outfit_prompt | llm

outfit_chain_with_history = RunnableWithMessageHistory(
    outfit_llm_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("[OK] Ba chain đã sẵn sàng")
print(f"QA prompt inputs     : {QA_PROMPT.input_variables}")
print(f"Outfit prompt inputs : {outfit_prompt.input_variables}")
print("Tiếp theo: chạy notebook 08 để xem router chọn chain nào.")


[THÔNG BÁO] Đang kết nối retriever và lắp ráp các chain...


: 